In [1]:
from pathlib import Path
import sys 
import os
import torch

In [2]:
# project root 설정
# Notebook에서는 직접 project root를 잡아 import 경로를 맞춘다.
candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path('/home/hyeonjin/detect-and-reason')]
PROJECT_ROOT = next(
    path for path in candidate_roots
    if (path / 'pyproject.toml').exists() and (path / 'src').exists()
)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /home/hyeonjin/detect-and-reason


In [3]:
# 학습 설정
print('torch.cuda.is_available():', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_count:', torch.cuda.device_count())
    print('current_device:', torch.cuda.current_device())
    print('device_name:', torch.cuda.get_device_name(torch.cuda.current_device()))

torch.cuda.is_available(): True
device_count: 7
current_device: 0
device_name: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition


In [5]:
from src.config_loader import load_yaml, resolve_runtime_config
from src.model.train import run_train
from src.model.predict import run_predict

In [6]:
from pathlib import Path

RESULT_ROOT = PROJECT_ROOT / 'notebook' / 'results' / 'detection_reasoning' / 'big'
RUNS_DIR = RESULT_ROOT / 'rt_detr_1cls'
PRED_DIR = RESULT_ROOT / 'rt_detr_1cls_prediction'
EVAL_DIR = RESULT_ROOT / 'rt_detr_1cls_eval'
OVERLAY_DIR = RESULT_ROOT / 'rt_detr_1cls_overlay'

In [8]:
train_cfg = resolve_runtime_config(
    model_ref='rt_detr_1cls',
    dataset_ref='tomatod',
    stage='train',
)

train_cfg['paths']['runs_dir'] = RUNS_DIR
train_cfg['train']['lr0'] = 0.001

print(train_cfg['paths'])
print(train_cfg['train'])

{'runs_dir': PosixPath('/home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls'), 'eval_dir': PosixPath('/home/hyeonjin/detect-and-reason/result/detection_reasoning/tomatod/rt_detr_1cls_eval'), 'prediction_dir': PosixPath('/home/hyeonjin/detect-and-reason/result/detection_reasoning/tomatod/rt_detr_1cls_prediction'), 'qwen_dir': PosixPath('/home/hyeonjin/detect-and-reason/result/detection_reasoning/tomatod/rt_detr_1cls_qwen'), 'weights': '/home/hyeonjin/detect-and-reason/rtdetr-l.pt'}
{'epochs': 150, 'imgsz': 640, 'batch': 12, 'workers': 8, 'run_name': 'rt_detr_1cls', 'augmentation': {'mosaic': 0.2, 'close_mosaic': 10, 'mixup': 0.1, 'scale': 0.2, 'translate': 0.1, 'degrees': 10, 'fliplr': 0.5, 'hsv_h': 0.015, 'hsv_s': 0.2, 'hsv_v': 0.2}, 'patience': 10, 'amp': True, 'optimizer': 'AdamW', 'lr0': 0.001, 'lrf': 0.01, 'cos_lr': True}


In [9]:
train_results = run_train(train_cfg)

New https://pypi.org/project/ultralytics/8.4.84 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 97258MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/hyeonjin/detect-and-reason/data/yolo/tomatod_1cls/data.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.2, hsv_v=0.2, imgsz=640, int8=False, iou=0.55, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, 

/home/hyeonjin/detect-and-reason/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1892.5±1329.6 MB/s, size: 190.2 KB)
val: Scanning /home/hyeonjin/detect-and-reason/data/yolo/tomatod_1cls/val/labels.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 1.2Mit/s 0.0s
WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
val: Caching images (0.0GB RAM): 100% ━━━━━━━━━━━━ 20/20 477.7it/s 0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 143 weight(decay=0.0), 206 weight(decay=0.00046875), 226 bias(decay=0.0)
Plotting labels to /home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls/labels.jpg... 
Image sizes 640 train, 640 val
Using 3 dataloader wor

/home/hyeonjin/detect-and-reason/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/150      9.29G      1.719       1.04     0.7478         61        640: 100% ━━━━━━━━━━━━ 19/19 1.6s/it 31.2s1.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.3s/it 2.3s
                   all         20        199    0.00833      0.251    0.00567    0.00126

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/150      9.86G      1.083      1.317      0.237         87        640: 0% ──────────── 0/19  0.7s

...

     30/150      9.86G     0.4543     0.7508    0.08656         63        640: 100% ━━━━━━━━━━━━ 19/19 1.5it/s 12.7s0.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.0it/s 0.3s
                   all         20        199      0.344      0.819      0.468      0.204

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     31/150      9.86G     0.4461     0.8198    0.07918        106        640: 0% ──────────── 0/19  0.5s

/home/hyeonjin/detect-and-reason/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     31/150      9.86G     0.4237     0.9033    0.07898         47        640: 100% ━━━━━━━━━━━━ 19/19 1.7it/s 11.5s0.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.4it/s 0.3s
                   all         20        199      0.228      0.864      0.387      0.215

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     32/150      9.86G     0.3866     0.9029    0.08439        135        640: 0% ──────────── 0/19  0.5s

/home/hyeonjin/detect-and-reason/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     32/150      9.86G     0.3811     0.8532    0.07017         91        640: 100% ━━━━━━━━━━━━ 19/19 1.0it/s 18.5s0.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.4it/s 0.3s
                   all         20        199      0.397      0.734      0.487      0.276
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 22, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

32 epochs completed in 0.147 hours.
Optimizer stripped from /home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls/weights/last.pt, 66.2MB
Optimizer stripped from /home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls/weights/best.pt, 66.2MB

Validating /home/hyeonjin/detect-and-reason/notebook/results/detection_reasonin

In [ ]:
# 추가적으로 설정 변경하고 싶은 것들 
# 머리 깨지도록 한 번 해보자. 누가 이기나. 

# config 설정
# model config 경로 로드
# model config의 override 부분을 원하는 대로 커스텀만 
# dataset config 경로 로드 
# 이후 train_run_cfg 생성 

In [10]:
best_weight = RUNS_DIR / 'weights' / 'best.pt'
print(best_weight, best_weight.exists())

/home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls/weights/best.pt True


In [12]:
predict_cfg = resolve_runtime_config(
    model_ref='rt_detr_1cls',
    dataset_ref='tomatod',
    stage='predict',
    weights=str(best_weight),
    source='test',
)

predict_cfg['paths']['prediction_dir'] = PRED_DIR
predict_cfg['predict']['save'] = False
predict_cfg['predict']['save_txt'] = True
predict_cfg['predict']['save_conf'] = True

print(predict_cfg['paths'])
predict_results = run_predict(predict_cfg)

{'runs_dir': PosixPath('/home/hyeonjin/detect-and-reason/runs/tomatod/rt_detr_1cls'), 'eval_dir': PosixPath('/home/hyeonjin/detect-and-reason/result/detection_reasoning/tomatod/rt_detr_1cls_eval'), 'prediction_dir': PosixPath('/home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls_prediction'), 'qwen_dir': PosixPath('/home/hyeonjin/detect-and-reason/result/detection_reasoning/tomatod/rt_detr_1cls_qwen'), 'weights': '/home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls/weights/best.pt'}

image 1/55 /home/hyeonjin/detect-and-reason/data/yolo/tomatod_1cls/test/images/stereo20190405_132743_p0_snap_514.jpg: 640x640 50 tomatos, 143.6ms
image 2/55 /home/hyeonjin/detect-and-reason/data/yolo/tomatod_1cls/test/images/stereo20190405_133623_p0_snap_043.jpg: 640x640 16 tomatos, 113.1ms
image 3/55 /home/hyeonjin/detect-and-reason/data/yolo/tomatod_1cls/test/images/stereo20190405_133623_p0_snap_089.jpg: 640x640 11 tomatos, 116.7ms
imag

In [13]:
eval_cfg = resolve_runtime_config(
    model_ref='rt_detr_1cls',
    dataset_ref='tomatod',
    stage='predict',
    weights=str(best_weight),
    source='test',
)

In [14]:
from pprint import pprint
from src.evaluation import (
    evaluate_yolo_label_predictions,
    save_detection_evaluation_artifacts,
)
from src.evaluation.yolo_label_io import (
    resolve_prediction_labels_dir,
    resolve_yolo_split_dirs,
)

EVAL_SPLIT = 'test'

# 노트북에서 prediction/eval 경로를 따로 쓰고 있으면 여기서 맞춰준다.
eval_cfg['paths']['prediction_dir'] = PRED_DIR
eval_cfg['paths']['eval_dir'] = EVAL_DIR

gt_images_dir, gt_labels_dir = resolve_yolo_split_dirs(
    eval_cfg['dataset'][f'{EVAL_SPLIT}_dir']
)

pred_labels_dir = resolve_prediction_labels_dir(
    eval_cfg['paths']['prediction_dir']
)

class_names = eval_cfg['class_names'] or eval_cfg['dataset']['class_names']

eval_results = evaluate_yolo_label_predictions(
    gt_images_dir=gt_images_dir,
    gt_labels_dir=gt_labels_dir,
    pred_labels_dir=pred_labels_dir,
    class_names=class_names,
    iou_threshold=eval_cfg['predict'].get('resolved_iou') or 0.5,
    weight_reference=eval_cfg['paths']['weights'],
    model_imgsz=int(eval_cfg.get('train', {}).get('imgsz', 640)),
)

artifact_paths = save_detection_evaluation_artifacts(
    eval_results,
    eval_cfg['paths']['eval_dir'],
)

stats = eval_results['detailed_statistics']['total_statistics']
det_metrics = eval_results['detection_metrics']

print('[evaluation summary]')
print('gt_labels_dir   :', gt_labels_dir)
print('pred_labels_dir :', pred_labels_dir)
print('output_dir      :', eval_cfg['paths']['eval_dir'])
print(f"precision         : {stats['overall_precision']:.4f}")
print(f"recall            : {stats['overall_recall']:.4f}")
print(f"f1                : {stats['overall_f1']:.4f}")
print(f"detection_acc     : {stats['detection_acc']:.4f}")
print(f"classification_acc: {stats['classification_acc']:.4f}")
print(f"overall_acc       : {stats['overall_acc']:.4f}")
print(f"mAP@0.5           : {det_metrics['map_50']:.4f}")
print(f"mAP@0.5:0.95      : {det_metrics['map']:.4f}")
print(f"mAP@0.75          : {det_metrics['map_75']:.4f}")
print(f"CA-mAP@0.5        : {det_metrics['ca_map_50']:.4f}")
print(f"CA-mAP@0.5:0.95   : {det_metrics['ca_map']:.4f}")

pprint(artifact_paths)

[evaluation summary]
gt_labels_dir   : /home/hyeonjin/detect-and-reason/data/yolo/tomatod_1cls/test/labels
pred_labels_dir : /home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls_prediction/labels
output_dir      : /home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls_eval
precision         : 0.2943
recall            : 0.9615
f1                : 0.4507
detection_acc     : 0.9615
classification_acc: 1.0000
overall_acc       : 0.9615
mAP@0.5           : 0.8196
mAP@0.5:0.95      : 0.5028
mAP@0.75          : 0.5815
CA-mAP@0.5        : 0.8196
CA-mAP@0.5:0.95   : 0.5028
{'confusion_csv': PosixPath('/home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls_eval/confusion_matrix.csv'),
 'json': PosixPath('/home/hyeonjin/detect-and-reason/notebook/results/detection_reasoning/big/rt_detr_1cls_eval/evaluation_results.json'),
 'per_class_csv': PosixPath('/home/hyeonjin/detect-and-reason/notebook/results/